# GDBA-Net Training (Google Colab)
Run cells in order. Upload your dataset and pretrained SAM2 checkpoint to the paths used below. Duplicate the last cell to train on another dataset.

In [1]:
!nvidia-smi
import sys, torch
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Mon Oct 12 23:48:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   61C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%cd /content
from pathlib import Path
import subprocess
repo = Path("/content/GDBA_Net")
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/biplabpoudel25/GDBA_Net.git", str(repo)], check=True)
else:
    print("Using existing GDBA_Net directory")
%cd /content/GDBA_Net

/content
Using existing GDBA_Net directory
/content/GDBA_Net


In [3]:
%pip install hydra-core tensorboardX einops PyYAML tqdm opencv-python PyWavelets scipy scikit-learn matplotlib Pillow

In [4]:
from pathlib import Path
REPO = Path("/content/GDBA_Net")
for relative in ["train.py", "models/sam.py", "configs/gdba_net.yaml"]:
    assert (REPO / relative).is_file(), f"Missing: {relative}"
print("Required repository files found.")

Required repository files found.


In [5]:
# The original Colab notebook bypassed unused legacy MMCV/MMseg imports.
from pathlib import Path
for relative in ["models/mmseg/__init__.py", "models/mmseg/models/__init__.py"]:
    file = REPO / relative
    if file.is_file():
        file.write_text("# Colab compatibility: bypass unused legacy MMseg imports\n")
        print("Updated:", relative)

Updated: models/mmseg/__init__.py
Updated: models/mmseg/models/__init__.py


In [6]:
# Create a Colab-specific YAML instead of modifying the repository original.
import yaml
original = REPO / "configs/gdba_net.yaml"
colab_config = REPO / "configs/gdba_colab_train.yaml"
with original.open() as f:
    config = yaml.safe_load(f)
for section in ["train_dataset", "val_dataset", "test_dataset"]:
    config[section]["batch_size"] = 1
    config[section]["num_workers"] = 0
with colab_config.open("w") as f:
    yaml.safe_dump(config, f, sort_keys=False)
print("Configuration:", colab_config.name)
for section in ["train_dataset", "val_dataset"]:
    print(section, "batch_size =", config[section]["batch_size"],
          "num_workers =", config[section]["num_workers"])

Configuration: gdba_colab_train.yaml
train_dataset batch_size = 1 num_workers = 0
val_dataset batch_size = 1 num_workers = 0


In [7]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Train
Edit the five input paths and experiment name. Copy this cell for additional runs.

In [8]:
%cd /content/GDBA_Net

!python train.py \
    --config configs/gdba_colab_train.yaml \
    --sam_ckpt "/content/drive/MyDrive/GDBA_Net_Colab/pretrained/sam2_hiera_large.pt" \
    --train_images /content/drive/MyDrive/GDBA_Net_Colab/datasets/MoNuSeg_80_20/train/images \
    --train_labels /content/drive/MyDrive/GDBA_Net_Colab/datasets/MoNuSeg_80_20/train/masks \
    --val_images /content/drive/MyDrive/GDBA_Net_Colab/datasets/MoNuSeg_80_20/val/images \
    --val_labels /content/drive/MyDrive/GDBA_Net_Colab/datasets/MoNuSeg_80_20/val/masks \
    --epochs 2 \
    --shot 100 \
    --name monuseg_200epochs

/content/GDBA_Net
Experiment name: monuseg_200epochs
cuda
Training dataset: 33 samples
Validation dataset: 7 samples
Inside handcrafted features
Number of trainable parameters: 4301098
Training: 100% 33/33 [01:40<00:00,  3.06s/it, loss=0.3701]
epoch 1/2, train G: loss=0.6674, val: dice=0.6888, val: iou=0.3353, 
New best checkpoint saved with dice = 0.6888, 2.3m 2.3m/4.6m
Training: 100% 33/33 [01:44<00:00,  3.17s/it, loss=0.3778]
epoch 2/2, train G: loss=0.4755, val: dice=0.6865, val: iou=0.4940, 2.2m 4.5m/4.5m
Training completed - Best model saved at epoch 1 with dice = 0.6888, Total training time: 205.5340 seconds
